# nib — Colab evaluation

**This notebook contains no logic.** It clones, installs, mounts Drive, copies two
files to local disk, and calls two scripts. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository, because the
moment logic moves into a notebook cell the run stops being reproducible.

## What this produces

The project's first real numbers, in two steps that must run **in this order**:

1. **What real handwriting scores on the line pack.** `check_metrics.py` measures
   the FID floor between two disjoint halves of real lines, the recogniser's own
   error rate, and the writer embedding's accuracy — then writes them to
   `references/references_cvl_lines_64.json`.
2. **What the generator scores against that.** `evaluate_generator.py` reads that
   file, so the baseline it reports against is one that was measured here rather
   than one retyped from a previous session.

Step 1 is cheap enough to run on a laptop. Step 2 is why this notebook wants a
GPU: generation is about 220 seconds per line on CPU, and minutes on a T4.

**Why not reuse the phase-1 numbers.** FID 33.72 and writer retrieval 66.9% were
measured on *word* crops. A line is five times wider and holds far more paper per
image, so it lands somewhere else entirely in Inception's feature space. Holding
a generated line against a word-level floor compares two different things.

## Before you start

Under `MyDrive/nib/` you need:

- **`cvl_lines_64.lmdb`** (148 MB). Upload the copy from
  `data/processed/upload/` — **never** the one in `data/processed/`, which LMDB
  reserves at 8 GB and which transfers as 8 GB.
- **`checkpoints/writer_embedder.pt`**. Without it, writer retrieval scores 3.7%
  on real handwriting and cannot tell a styled generator from an unstyled one.

## 1. Clone and install

`torch` is deliberately not in the base dependencies — Colab's build is matched to
its CUDA driver, and installing ours on top can replace a working build with one
that does not match the GPU.

The `models` extra **is** installed, and its `transformers<5` pin is
load-bearing: Emuru's own code defines `_tied_weights_keys` while 5.x looks for
`all_tied_weights_keys`, so 5.x cannot load the model at all. The same gap breaks
TrOCR's tokenizer.

If pip replaces a `transformers` that was already imported, Colab will ask you to
**restart the runtime**. Do it, then carry on from cell 2 — nothing above needs
re-running except the clone, which is a no-op the second time.

In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track,models]"

## 2. What are we running on

Record this. If Colab's versions differ from the local ones, the same code can
behave differently in the two places — and the `transformers` major version is
the one that has already cost this project a day.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
import transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, " <- must be 4.x")
assert transformers.__version__.startswith("4."), "5.x cannot load Emuru; see pyproject"

## 3. Mount Drive and copy what the run needs

**The copy is the point.** Reading the pack record-by-record over Drive would
leave the GPU waiting on network round-trips. One sequential copy of one file,
and every read after it is local.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed /content/nib/checkpoints
!time cp /content/drive/MyDrive/nib/cvl_lines_64.lmdb /content/nib/data/processed/
!cp /content/drive/MyDrive/nib/checkpoints/writer_embedder.pt /content/nib/checkpoints/
!ls -lh /content/nib/data/processed/ /content/nib/checkpoints/

## 4. Is everything here

One command, and it answers rather than reassures. The word pack will be reported
missing and that is fine — one pack is required, not both, and this session only
needs lines.

In [ ]:
!python scripts/check_data.py

## 5. What real handwriting scores (T15)

**Already measured on CPU and committed** as
`references/references_cvl_lines_64.json`:

```
FID floor        19.06     two disjoint halves of real lines   (words: 33.72)
writer top-1     83.7%     the embedding on real lines         (words: 66.9%)
writer top-5     97.8%
CER              13.36%    TrOCR's own error rate on real lines
```

Both of the first two moved a long way from their word-level values, which is
exactly why they had to be re-measured: against the old floor a generated set
would have looked almost twice as good as it is, and the retrieval bar is far
higher than it appeared.

Re-run it here anyway, for one reason: the committed CER came from **40** lines,
which is a small enough sample to move by a point or two. `--cer-lines 300`
tightens it. If FID and retrieval come out differently on GPU than the CPU
numbers above, that is itself a finding worth reporting.

Read rather than skip: **FID(real, same real)** must be about 0. Anything else
means the feature extraction is broken and no other FID here means anything.

In [ ]:
!python scripts/check_metrics.py \
    --pack data/processed/cvl_lines_64.lmdb \
    --samples 300 \
    --cer-lines 300 \
    --device cuda

## 6. Generate, and score it (T16)

300 lines in held-out writers' hands — 94 writers no model in this project has
trained on — each one a line that writer really wrote, so a real image of exactly
that text in exactly that hand exists to compare against. The generator never
sees it.

Watch the **truncation count**. Emuru stops when it decides the line is finished;
anything that instead runs to its token budget comes out cut short, and its CER
is charged for the missing ending. A few are the known non-stopping failure. A
lot means `TOKENS_PER_CHAR` in `src/nib/models/emuru.py` is too low.

In [ ]:
!python scripts/evaluate_generator.py \
    --generator emuru \
    --unit lines \
    --samples 300 \
    --style-refs 1 \
    --batch-size 8 \
    --device cuda

## 7. Keep the results

The VM's local disk dies with the session.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines /content/drive/MyDrive/nib/results/
!cp -r /content/nib/references /content/drive/MyDrive/nib/results/
!ls -lh /content/drive/MyDrive/nib/results/

# Printed as well as copied: this is the file that belongs in git, and reading it
# here is faster than fetching it off Drive to see whether the numbers moved.
!cat /content/nib/references/references_cvl_lines_64.json
!cat /content/nib/outputs/eval_emuru_lines/results.json

## What to report back

- `transformers` version from cell 2, and which GPU you got
- The three reference numbers from cell 5, and the line reader's drop report
- The summary block from cell 6: FID, writer top-1, CER gap, truncation count
- Generation rate in lines per second, so the next run can be planned
- Anything that failed, with the error text

The reference numbers matter most. Until they exist, every result this project
has is a number without a scale.